# GBM RTA: public analysis notebook

Private study inputs are not distributed. Set GBM_RTA_ROOT to your authorized local data directory. The clinical workbook placeholder is clinical_metadata.xlsx. Patient-specific examples and all saved outputs have been removed; do not commit outputs generated with private data.


In [ ]:
import pandas as pd
from scipy.stats import mannwhitneyu, fisher_exact, pearsonr, spearmanr, kendalltau, wilcoxon
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score
import os
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import textwrap
from tqdm import tqdm
import subprocess
from sklearn.metrics import roc_curve, auc
from sklearn.cluster import KMeans
import ants
from lifelines import KaplanMeierFitter, CoxPHFitter, utils
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index # C-index
from matplotlib.offsetbox import AnnotationBbox, TextArea, HPacker, VPacker
from matplotlib import cm
from sklearn.metrics import confusion_matrix
from statsmodels.stats.multitest import multipletests

In [ ]:
root = os.environ.get("GBM_RTA_ROOT", "./private_data")


In [ ]:
data_dir = f'{root}/data/only_initial'

In [ ]:
palette  = {key:value for key, value in enumerate(plt.get_cmap("tab10").colors)}

In [ ]:
original_meta = pd.read_excel(f'{root}/excel_csv/clinical_metadata.xlsx')
original_meta = original_meta[['환자번호', '성별', '처음수술당시나이', 'Telomerase activity1', 'Telomerase activity2', 'RTA△group (Extend score)', 'TERTp', 'MGMTp', 'CCRT', 'OP1', 'TERTp.1', 'MGMTp.1', 'OP2', 'PFS(W)', 'OS(W)', 'PFS(m)', 'OS1(m)', 'OS2(m)', '사망일']]
original_meta.rename({'환자번호':'ID', '성별':'sex', '처음수술당시나이':'age', 'RTA△group (Extend score)': 'RTA', 'Telomerase activity1':'TA1', 'Telomerase activity2':'TA2', '사망일':'event_observed', 'TERTp.1':'TERTp2', 'MGMTp.1':'MGMTp2'}, axis=1, inplace=True)
original_meta.set_index('ID', inplace=True)
original_meta.index = original_meta.index.astype(str)
original_meta['RTA_cont'] = original_meta['TA2'] - original_meta['TA1']
original_meta['event_observed'] = original_meta['event_observed'].map(lambda x: 0 if pd.isna(x) else 1)
original_meta['decrease'] = original_meta['RTA'].map(lambda x: 1 if x == 'D' else 0)
original_meta['increase'] = original_meta['RTA'].map(lambda x: 1 if x == 'I' else 0)

original_meta['TERTp'] = (original_meta['TERTp']=='mut').astype(int)
original_meta['MGMTp'] = (original_meta['MGMTp']=='methylation').astype(int)
original_meta['TERTp2'] = (original_meta['TERTp']=='mut').astype(int)
original_meta['MGMTp2'] = (original_meta['MGMTp']=='methylation').astype(int)
original_meta.CCRT = original_meta.CCRT.replace('O(Gy모름)', np.nan)
original_meta = original_meta.loc[original_meta.index.astype(str).isin(os.listdir(data_dir))]

original_meta = original_meta.loc[original_meta['OP1']=='GTR']
print(f'length of original_meta: {len(original_meta)}')
original_meta.columns

In [ ]:
print('\nUnivariate1')
cph = CoxPHFitter()
for survival_target in ['OS(W)', 'PFS(W)']:
    print(survival_target)
    for data in ['TA1', 'TERTp', 'MGMTp']:
        df = original_meta[[survival_target, 'event_observed', data]].copy() 
        cph.fit(df, duration_col=survival_target, event_col='event_observed')
        df['partial_hazard'] = cph.predict_partial_hazard(df)
        c_index = concordance_index(df[survival_target], -df['partial_hazard'], df['event_observed'])
        print(f'{data} p-val: {round(cph.summary["p"][data], 3)}, HR ratio: {cph.summary["exp(coef)"][data]}')

In [ ]:
print('\nMultivariate1')
cph = CoxPHFitter()
for survival_target in ['OS(W)', 'PFS(W)']:
    df = original_meta[[survival_target, 'event_observed', 'TERTp', 'MGMTp']].copy() 
    cph.fit(df, duration_col=survival_target, event_col='event_observed')
    df['partial_hazard'] = cph.predict_partial_hazard(df)
    c_index = concordance_index(df[survival_target], -df['partial_hazard'], df['event_observed'])
    print(f'{survival_target} C-index: {c_index:.4f}')
    variables = ['TERTp', 'MGMTp']
    bootstrap_results = {var: [] for var in variables}
    for v in variables:
        print(f'{v} p-val: {cph.summary["p"][v]}, HR ratio: {cph.summary["exp(coef)"][v]}')

In [ ]:
print('\nMultivariate1')
cph = CoxPHFitter()
for survival_target in ['OS(W)', 'PFS(W)']:
    df = original_meta[[survival_target, 'event_observed', 'TA1', 'TERTp', 'MGMTp']].copy() 
    cph.fit(df, duration_col=survival_target, event_col='event_observed')
    df['partial_hazard'] = cph.predict_partial_hazard(df)
    c_index = concordance_index(df[survival_target], -df['partial_hazard'], df['event_observed'])
    print(f'{survival_target} C-index: {c_index:.4f}')
    variables = ['TA1', 'TERTp', 'MGMTp']
    bootstrap_results = {var: [] for var in variables}
    for v in variables:
        print(f'{v} p-val: {cph.summary["p"][v]}, HR ratio: {cph.summary["exp(coef)"][v]}')

In [ ]:
print('\nMultivariate1')
cph = CoxPHFitter()
for survival_target in ['OS(W)', 'PFS(W)']: 
    df = original_meta[[survival_target, 'event_observed', 'TA1', 'MGMTp']].copy() 
    cph.fit(df, duration_col=survival_target, event_col='event_observed')
    df['partial_hazard'] = cph.predict_partial_hazard(df)
    c_index = concordance_index(df[survival_target], -df['partial_hazard'], df['event_observed'])
    print(f'{survival_target} C-index: {c_index:.4f}')
    print(f'TA1 p-val: {cph.summary["p"]["TA1"]}, HR ratio: {cph.summary["exp(coef)"]["TA1"]}')
    # print(f'TERTp p-val: {cph.summary["p"]["TERTp"]}, HR ratio: {cph.summary["exp(coef)"]["TERTp"]}')
    print(f'MGMTp p-val: {cph.summary["p"]["MGMTp"]}, HR ratio: {cph.summary["exp(coef)"]["MGMTp"]}')

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(14, 6))


survival_target = 'OS(W)'
df = original_meta[[survival_target, 'event_observed', 'TA1']].copy()
cph.fit(df, duration_col=survival_target, event_col='event_observed')
df['partial_hazard'] = cph.predict_partial_hazard(df)
median_hazard = 35
high_hazard = df[df['TA1'] > median_hazard].copy() 
low_hazard = df[df['TA1'] <= median_hazard].copy() 

kmf_D = KaplanMeierFitter()
kmf_ND = KaplanMeierFitter()
kmf_D.fit(high_hazard[survival_target], high_hazard['event_observed'], label=f'high_TA1 (n={len(high_hazard)})')
kmf_ND.fit(low_hazard[survival_target], low_hazard['event_observed'], label=f'low_TA1 (n={len(low_hazard)})')

kmf_D.plot_survival_function(ax=axs[0])
kmf_ND.plot_survival_function(ax=axs[0])
results = logrank_test(high_hazard[survival_target], low_hazard[survival_target],
                       event_observed_A=high_hazard['event_observed'],
                       event_observed_B=low_hazard['event_observed'])
axs[0].set_title(f"Kaplan-Meier Survival Curves based on Overall Survival\nLog-rank p={results.p_value:.3f}")
axs[0].set_xlabel("Weeks")
axs[0].set_ylabel("Survival Probability")


survival_target = 'PFS(W)'
df = original_meta[[survival_target, 'event_observed', 'TA1']].copy()
cph.fit(df, duration_col=survival_target, event_col='event_observed')
df['partial_hazard'] = cph.predict_partial_hazard(df)
high_hazard = df[df['TA1'] > median_hazard].copy() 
low_hazard = df[df['TA1'] <= median_hazard].copy() 

kmf_D = KaplanMeierFitter()
kmf_ND = KaplanMeierFitter()
kmf_D.fit(high_hazard[survival_target], high_hazard['event_observed'], label=f'high_TA1 (n={len(high_hazard)})')
kmf_ND.fit(low_hazard[survival_target], low_hazard['event_observed'], label=f'low_TA1 (n={len(low_hazard)})')

kmf_D.plot_survival_function(ax=axs[1])
kmf_ND.plot_survival_function(ax=axs[1])
results = logrank_test(high_hazard[survival_target], low_hazard[survival_target],
                       event_observed_A=high_hazard['event_observed'],
                       event_observed_B=low_hazard['event_observed'])
axs[1].set_title(f"Kaplan-Meier Survival Curves based on Progression-Free Survival\nLog-rank p={results.p_value:.3f}")
axs[1].set_xlabel("Weeks")
axs[1].set_ylabel("Survival Probability")

plt.tight_layout()
plt.legend()
# plt.savefig(f"{root}/figures/FigureS1_TA_survival.png", format="png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
survival_target = 'OS(W)' 
survival = original_meta[[survival_target, 'event_observed', 'TA1', 'TA2', 'TERTp', 'MGMTp']] 
survival = survival.merge(original_meta[['sex', 'age', 'RTA', 'RTA_cont', 'OP1']], left_index=True, right_index=True)
survival['high_TA1'] = 0

survival.loc[survival['TA1']>35, 'high_TA1'] = 1
high_TA1 = survival[survival['high_TA1']==1]
low_TA1 = survival[survival['high_TA1']==0]

increase = survival[survival['RTA']=='I']
non_increase = survival[survival['RTA']!='I']

decrease = survival[survival['RTA']=='D']
non_decrease = survival[survival['RTA']!='D']

# Participant-level table display omitted for public distribution.

In [ ]:
print('Sex')
print('TA1:', fisher_exact([[len(high_TA1[high_TA1['sex']=='M']), len(high_TA1[high_TA1['sex']!='M'])], 
              [len(low_TA1[low_TA1['sex']=='M']), len(low_TA1[low_TA1['sex']!='M'])]]))
print('decrease:', fisher_exact([[len(decrease[decrease['sex']=='M']), len(decrease[decrease['sex']!='M'])], 
              [len(non_decrease[non_decrease['sex']=='M']), len(non_decrease[non_decrease['sex']!='M'])]]))

In [ ]:
print('TERTp mut')
print('TA1:', fisher_exact([[len(high_TA1[high_TA1['TERTp']==1]), len(high_TA1[high_TA1['TERTp']!=1])],
              [len(low_TA1[low_TA1['TERTp']==1]), len(low_TA1[low_TA1['TERTp']!=1])]]))
print('decrease:', fisher_exact([[len(decrease[decrease['TERTp']==1]), len(decrease[decrease['TERTp']!=1])], 
              [len(non_decrease[non_decrease['TERTp']==1]), len(non_decrease[non_decrease['TERTp']!=1])]]))

In [ ]:
print('MGMTp methylation')
print('TA1:', fisher_exact([[len(high_TA1[high_TA1['MGMTp']==1]), len(high_TA1[high_TA1['MGMTp']!=1])], 
                            [len(low_TA1[low_TA1['MGMTp']==1]), len(low_TA1[low_TA1['MGMTp']!=1])]]))
print('decrease:', fisher_exact([[len(decrease[decrease['MGMTp']==1]), len(decrease[decrease['MGMTp']!=1])], 
              [len(non_decrease[non_decrease['MGMTp']==1]), len(non_decrease[non_decrease['MGMTp']!=1])]]))

In [ ]:
print('Age')
print('TA1:', mannwhitneyu(high_TA1.age, low_TA1.age))
print('decrease:', mannwhitneyu(decrease.age, non_decrease.age))

In [ ]:
print('TA1')
print('TA1:', mannwhitneyu(high_TA1.TA1, low_TA1.TA1))
print('decrease:', mannwhitneyu(decrease.TA1, non_decrease.TA1))

In [ ]:
print('TA2')
print('TA1:', mannwhitneyu(high_TA1.TA2, low_TA1.TA2))
print('decrease:', mannwhitneyu(decrease.TA2, non_decrease.TA2))

In [ ]:
mannwhitneyu(original_meta.loc[original_meta['decrease']==1].CCRT.dropna(),
original_meta.loc[original_meta['decrease']==0].CCRT)

In [ ]:
pre_total = pd.DataFrame()
for subj in tqdm(sorted(original_meta.index)):
    directory = os.path.join(data_dir, subj, '1')

    df = pd.DataFrame()
    for file_name, name in zip(['flair_vox.nii.gz', 'rCBV_vox.nii.gz', 'adc_vox.nii.gz', 't1ce_vox.nii.gz', 
                                'flair_rad.nii.gz', 'rCBV_rad.nii.gz', 'adc_rad.nii.gz', 't1ce_rad.nii.gz', 
                                ], 
                         ['flair', 'cbv', 'adc', 't1ce', 
                          'flair_normed', 'cbv_normed', 'adc_normed', 't1ce_normed', 
                          ]):
        img = ants.image_read(os.path.join(directory, file_name))
        img = img.numpy()
        img = img.flatten()
        df = pd.concat([df, pd.DataFrame({name:img})], axis=1)

    for file_name, name in zip(['NE_tumor_mask.nii.gz', 'CE_tumor_mask.nii.gz'], ['NE', 'CE']):
        img = ants.image_read(os.path.join(directory, file_name))
        img = img.numpy()
        img = img.flatten()
        img = pd.Series(img)
        img = img[img>0].index
        df[name] = None
        df.loc[img, name] = img

    df = df[(~pd.isnull(df['NE'])) | (~pd.isnull(df['CE']))]
    df['subj'] = subj
    df['flair_adc'] = (df['flair_normed']+10000) / (df['adc_normed']+10000)
    df['cbv_adc'] = (df['cbv_normed']+10000) / (df['adc_normed']+10000)

    pre_total = pd.concat([pre_total, df], axis=0)
    
pre_total['NE1CE0'] = pre_total.apply(lambda row: 1 if row['NE'] is not None else 0, axis=1)
# leave only valid voxels
pre_total = pre_total[(pre_total['adc']>0)&(pre_total['cbv']>0)&(pre_total['flair']>0)&(pre_total['t1ce']>0)]
# 99.9 percentile as outliers
percentiles = pre_total[['adc', 'cbv', 'flair', 't1ce', 'flair_adc', 'cbv_adc']].quantile(0.999)
# remove outlier
pre_total = pre_total[
    (pre_total['adc'] <= percentiles['adc']) &
    (pre_total['cbv'] <= percentiles['cbv']) &
    (pre_total['flair'] <= percentiles['flair']) &
    (pre_total['t1ce'] <= percentiles['t1ce']) # &
]

total_range = {'cbv_min': pre_total['cbv_normed'].min(), 
               'cbv_max': pre_total['cbv_normed'].max(),
               'adc_min': pre_total['adc_normed'].min(), 
               'adc_max': pre_total['adc_normed'].max()}

common_range = {'cbv_min': total_range['cbv_min'], 
               'cbv_max': total_range['cbv_max'],
               'adc_min': total_range['adc_min'], 
               'adc_max': total_range['adc_max']}

for subj in tqdm(sorted(original_meta.index)):
    if not os.path.isdir(os.path.join(data_dir, subj)): continue

    df = pre_total[pre_total['subj']==subj]
    cbv_min = df.cbv_normed.min()
    if cbv_min > common_range['cbv_min']:
        common_range['cbv_min'] = cbv_min

    cbv_max = df.cbv_normed.max()
    if cbv_max < common_range['cbv_max']:
        common_range['cbv_max'] = cbv_max

    adc_min = df.adc_normed.min()
    if adc_min > common_range['adc_min']:
        common_range['adc_min'] = adc_min

    adc_max = df.adc_normed.max()
    if adc_max < common_range['adc_max']:
        common_range['adc_max'] = adc_max


In [ ]:
n_cluster=3
df = pre_total[(pre_total['cbv_normed']>common_range['cbv_min'])&
               (pre_total['adc_normed']>common_range['adc_min'])&
               (pre_total['cbv_normed']<common_range['cbv_max'])&
               (pre_total['adc_normed']<common_range['adc_max'])]

kmeans = KMeans(n_clusters = n_cluster, random_state=2 # 123
                      )

kmeans.fit(df[['cbv_normed', 'adc_normed']]) 

pre_wt_center = kmeans.cluster_centers_

pre_total['wt_kmeans'] = kmeans.predict(pre_total[['cbv_normed', 'adc_normed']])
sns.scatterplot(data=pre_total, x='cbv_normed', y='adc_normed', hue='wt_kmeans', s=1, palette=palette)
plt.title(f'Whole tumor clustering {len(pre_total)} voxels')
plt.xlabel('nrCBV')
plt.ylabel('nADC')
plt.legend(markerscale=5)
plt.plot([common_range['cbv_min'], common_range['cbv_min'], common_range['cbv_max'], common_range['cbv_max'], common_range['cbv_min']], 
         [common_range['adc_min'], common_range['adc_max'], common_range['adc_max'], common_range['adc_min'], common_range['adc_min']], 
         'r-', 
         linewidth=2)
plt.xlim(total_range['cbv_min']-50, total_range['cbv_max']+50)
plt.ylim(total_range['adc_min']-50, total_range['adc_max']+50)

In [ ]:
total = {}
total['initial'] = pre_total

In [ ]:
from matplotlib.font_manager import FontProperties
font_prop = FontProperties(family='Ubuntu', size=14, weight='bold')
row, col = 1, 1 
fig, ax = plt.subplots(nrows=row, ncols=col, figsize=(col*7, row*5))
plt.subplots_adjust(wspace=0.4, hspace=0.4)

xlim = (total_range['cbv_min'], total_range['cbv_max'])
ylim = (total_range['adc_min'], total_range['adc_max'])

subjs = ['initial']
titles = ['Initial', 'Recur']

for i, subj in enumerate(subjs):
    pre_to_plot = total[subj].copy()
    pre_to_plot['wt_kmeans_named'] = pre_to_plot['wt_kmeans'].map({0:'Nonviable', 
                                                                   1: 'Hypovascular cellular',
                                                                   2: 'Hypervascular cellular'})
    sns.scatterplot(
        data=pre_to_plot,
        x='cbv_normed',
        y='adc_normed',
        s=1,
        hue='wt_kmeans_named',
        hue_order=['Nonviable', 'Hypovascular cellular', 'Hypervascular cellular'],
        palette={'Nonviable':palette[0], 'Hypovascular cellular':palette[1], 'Hypervascular cellular':palette[2]},
        ax=ax
    ).set(xlim=xlim, ylim=ylim)
    ax.legend(markerscale=5)
    ax.set_title(f'{titles[i]} whole tumor clustering {len(pre_to_plot)} voxels')
    ax.set_xlabel('nrCBV', fontsize=10, fontweight='bold')
    ax.set_ylabel('nADC', fontsize=10, fontweight='bold')

plt.subplots_adjust(hspace=0.2, wspace=0.3) 
# plt.savefig(f"{root}/figures/Figure1_clustering.png", format="png", dpi=dpi, bbox_inches='tight')
plt.show()


In [ ]:
cmap = cm.get_cmap('nipy_spectral')  

row, col = 1, 2 
fig, ax = plt.subplots(nrows=row, ncols=col, figsize=(col*7, row*5))
plt.subplots_adjust(wspace=0.4, hspace=0.4)

xlim = (total_range['cbv_min'], total_range['cbv_max'])
ylim = (total_range['adc_min'], total_range['adc_max'])

subjs = [1, 0]
titles = ['Non-enhancing tumor', 'Enhancing tumor']

for i, subj in enumerate(subjs):
    pre_to_plot = total['initial'].loc[total['initial']['NE1CE0']==subj].copy() 
    pre_to_plot['wt_kmeans_named'] = pre_to_plot['wt_kmeans'].map({0:'Nonviable', 
                                                                   1: 'Hypovascular cellular',
                                                                   2: 'Hypervascular cellular'})
    sns.scatterplot(
        data=pre_to_plot,
        x='cbv_normed',
        y='adc_normed',
        s=1,
        hue='wt_kmeans_named',
        hue_order=['Nonviable', 'Hypovascular cellular', 'Hypervascular cellular'],
        palette={'Nonviable':palette[0], 'Hypovascular cellular':palette[1], 'Hypervascular cellular':palette[2]},
        ax=ax[i]
    ).set(title=f'{titles[i]} {len(pre_to_plot)} voxels', xlim=xlim, ylim=ylim)
    ax[i].legend(markerscale=5)
    ax[i].set_xlabel('nrCBV')
    ax[i].set_ylabel('nADC')

plt.subplots_adjust(hspace=0.2, wspace=0.3) 
plt.show()


In [ ]:
pre_total = total['initial'].copy()
voxel_ratio = pd.DataFrame()
for subj in tqdm(sorted(original_meta.index)):

    rta = survival.loc[subj].RTA
    rta_cont = survival.loc[subj].RTA_cont
    ta1 = survival.loc[subj].TA1
    sex = survival.loc[subj].sex
    age = survival.loc[subj].age
    
    if sex == 'M':
        sex = 1
    else: # age == 'F':
        sex = 0

    if rta == 'D':
        increase=0
        decrease=1
    elif rta == 'I':
        increase=1
        decrease=0
    else: # if rta == 'S':
        increase=0
        decrease=0
        
    subj_df = pre_total[pre_total['subj']==subj]
    dictionary = {'RTA':rta, 'RTA_cont':rta_cont, 'TA1':ta1, 'increase':increase, 'decrease':decrease, 'Sex':sex, 'Age':age}
    for region in ['wt', 'ne', 'ce']:
        for cluster in range(n_cluster):
            if region == 'wt':
                dictionary[f'{region}_voxel_num'] = len(subj_df)
                dictionary[f'{region}_{cluster}_voxel_num'] = len(subj_df[subj_df['wt_kmeans'] == cluster])
                dictionary[f'{region}_{cluster}'] = dictionary[f'{region}_{cluster}_voxel_num']/dictionary[f'{region}_voxel_num']
            elif region == 'ne':
                dictionary[f'{region}_voxel_num'] = len(subj_df[subj_df['NE1CE0']==1])
                dictionary[f'{region}_{cluster}_voxel_num'] = len(subj_df[(subj_df['NE1CE0']==1)&(subj_df['wt_kmeans'] == cluster)])
                dictionary[f'{region}_{cluster}'] = dictionary[f'{region}_{cluster}_voxel_num']/dictionary[f'{region}_voxel_num']
            elif region == 'ce':
                dictionary[f'{region}_voxel_num'] = len(subj_df[subj_df['NE1CE0']==0])
                dictionary[f'{region}_{cluster}_voxel_num'] = len(subj_df[(subj_df['NE1CE0']==0)&(subj_df['wt_kmeans'] == cluster)])
                dictionary[f'{region}_{cluster}'] = dictionary[f'{region}_{cluster}_voxel_num']/dictionary[f'{region}_voxel_num']

    voxel_ratio = pd.concat([voxel_ratio, pd.DataFrame(dictionary, index=[int(subj)])], axis=0)


In [ ]:
def calc_volume_ml(patient_dir): # study_dir
    try:
        ne = subprocess.run(["fslstats", f"{patient_dir}/NE_tumor_mask.nii.gz", "-V"], capture_output=True).stdout
        ne = float(ne.decode().split(' ')[1])/1000
    except Exception as e:
        print("Volume calculation failed for a private input.")
        print(type(e).__name__)
        ne=None

    try:
        et = subprocess.run(["fslstats", f"{patient_dir}/CE_tumor_mask.nii.gz", "-V"], capture_output=True).stdout
        et = float(et.decode().split(' ')[1])/1000
    except Exception as e:
        print("Volume calculation failed for a private input.")
        print(type(e).__name__)
        et=None

    try:
        wt = subprocess.run(["fslstats", f"{patient_dir}/WT_tumor_mask.nii.gz", "-V"], capture_output=True).stdout
        wt = float(wt.decode().split(' ')[1])/1000
    except Exception as e:
        print("Volume calculation failed for a private input.")
        print(type(e).__name__)
        wt=None

    try:
        brain = subprocess.run(["fslstats", f"{patient_dir}/t1ce_vox.nii.gz", "-l", "0", "-V"], capture_output=True).stdout
        brain = float(brain.decode().split(' ')[1])/1000
    except Exception as e:
        print("Volume calculation failed for a private input.")
        print(type(e).__name__)
        brain=None

    pid = patient_dir.split('/')[-2]
    return pd.DataFrame({'subj': pid,
                            'brain_volume': brain,
                            'wt_volume': wt,
                            'ce_volume': et, 
                            'ne_volume': ne, 
                            'study_path': patient_dir, 
                            }, index=[pid])



output = pd.DataFrame()
for sd in [f'{data_dir}/{pid}/1' for pid in survival.index]:
    df = calc_volume_ml(sd)
    output = pd.concat([output, df], axis=0)

output = output[['brain_volume', 'wt_volume', 'ce_volume', 'ne_volume']]

output.index = output.index.astype(int)
voxel_ratio = voxel_ratio.join(output, how='left')
voxel_ratio.index.name = 'ID'

# Participant-level table display omitted for public distribution.

In [ ]:
habitat_ratio = {}
habitat_ratio['initial'] = voxel_ratio.copy()

In [ ]:
for op in ['initial']: 
    voxel_ratio = habitat_ratio[op].copy()
    for region in ['wt', 'ne', 'ce']:
        for cluster in range(n_cluster):
            voxel_ratio[f'{region}_{cluster}_volume'] = voxel_ratio[f'{region}_{cluster}'] * voxel_ratio[f'{region}_volume']
            voxel_ratio[f'{region}_{cluster}_adj'] = voxel_ratio[f'{region}_{cluster}_volume'] / voxel_ratio['brain_volume']
    habitat_ratio[op] = voxel_ratio.copy()

In [ ]:
print('Univariable')
cph = CoxPHFitter()
voxel_ratio = habitat_ratio['initial'].copy()

for survival_target in ['PFS(W)', 'OS(W)']: 
    column = ['ne_1_volume'] 
    df = voxel_ratio[column] 
    df.index = df.index.astype(str)
    df = df.merge(original_meta[[survival_target, 'event_observed']], left_index=True, right_index=True)
    cph.fit(df, duration_col=survival_target, event_col='event_observed')
    df['partial_hazard'] = cph.predict_partial_hazard(df)
    c_index = concordance_index(df[survival_target], -df['partial_hazard'], df['event_observed'])
    print(f'{survival_target} C-index: {c_index:.4f}')
    for c in column:
        print(f'{c} p-val: {cph.summary["p"][c]}, HR ratio: {cph.summary["exp(coef)"][c]}')


In [ ]:
for region in ['ne']:
    for cluster in [1]:
        for data in [f'{region}_{cluster}_volume']:
            for survival_target in ['PFS(W)', 'OS(W)']: 
                column = [data] 
                df = voxel_ratio[column] 
                df.index = df.index.astype(str)
                df = df.merge(original_meta[[survival_target, 'event_observed']], left_index=True, right_index=True)
                cph.fit(df, duration_col=survival_target, event_col='event_observed')
                df['partial_hazard'] = cph.predict_partial_hazard(df)
                c_index = concordance_index(df[survival_target], -df['partial_hazard'], df['event_observed'])
                print(f'{survival_target} C-index: {c_index:.4f}')
                for c in column:
                    p_val = cph.summary["p"][c]
                    if p_val < 0.05:
                        print(f'{c} p-val: {p_val}, HR ratio: {cph.summary["exp(coef)"][c]}')
            print()

In [ ]:
print('\nMultivariable1')
cph = CoxPHFitter()
voxel_ratio = habitat_ratio['initial'].copy()
for survival_target in ['PFS(W)', 'OS(W)']: 
    column = ['ne_1_volume', 'TA1']
    df = voxel_ratio[column] 
    df.index = df.index.astype(str)
    df = df.merge(original_meta[[survival_target, 'event_observed']], left_index=True, right_index=True)
    cph.fit(df, duration_col=survival_target, event_col='event_observed')
    df['partial_hazard'] = cph.predict_partial_hazard(df)
    c_index = concordance_index(df[survival_target], -df['partial_hazard'], df['event_observed'])
    print(f'{survival_target} C-index: {c_index:.4f}')
    for c in column:
        print(f'{c} p-val: {cph.summary["p"][c]}, HR ratio: {cph.summary["exp(coef)"][c]}')


In [ ]:
print('\nMultivariate1')
cph = CoxPHFitter()
voxel_ratio = habitat_ratio['initial'].copy()
voxel_ratio.index = voxel_ratio.index.astype(str)
voxel_ratio = voxel_ratio.join(original_meta[['MGMTp']], how='left')
for survival_target in ['PFS(W)', 'OS(W)']:
    column = ['ne_1_volume', 'MGMTp'] 
    df = voxel_ratio[column] 
    df.index = df.index.astype(str)
    df = df.merge(original_meta[[survival_target, 'event_observed']], left_index=True, right_index=True)
    cph.fit(df, duration_col=survival_target, event_col='event_observed')
    df['partial_hazard'] = cph.predict_partial_hazard(df)
    c_index = concordance_index(df[survival_target], -df['partial_hazard'], df['event_observed'])
    print(f'{survival_target} C-index: {c_index:.4f}')
    for c in column:
        print(f'{c} p-val: {cph.summary["p"][c]}, HR ratio: {cph.summary["exp(coef)"][c]}')

In [ ]:
print('\nMultivariate1')
cph = CoxPHFitter()
voxel_ratio = habitat_ratio['initial'].copy()
voxel_ratio.index = voxel_ratio.index.astype(str)
voxel_ratio = voxel_ratio.join(original_meta[['TERTp']], how='left')
for survival_target in ['PFS(W)', 'OS(W)']: 
    column = ['ne_1_volume', 'TERTp'] 
    df = voxel_ratio[column] 
    df.index = df.index.astype(str)
    df = df.merge(original_meta[[survival_target, 'event_observed']], left_index=True, right_index=True)
    cph.fit(df, duration_col=survival_target, event_col='event_observed')
    df['partial_hazard'] = cph.predict_partial_hazard(df)
    c_index = concordance_index(df[survival_target], -df['partial_hazard'], df['event_observed'])
    print(f'{survival_target} C-index: {c_index:.4f}')
    for c in column:
        print(f'{c} p-val: {cph.summary["p"][c]}, HR ratio: {cph.summary["exp(coef)"][c]}')

In [ ]:
voxel_ratio = habitat_ratio['initial'].copy()
voxel_ratio.index = voxel_ratio.index.astype(str)
voxel_ratio = voxel_ratio.merge(survival[['high_TA1', 'OP1']], left_index=True, right_index=True)
treshold = 0.05

In [ ]:
target = 'high_TA1'
y = voxel_ratio[target]
print('===========================================================================')
print(target)
print(f'number of {target}: ', len(voxel_ratio.loc[voxel_ratio[target]==1]))
print(f'number of non-{target}: ', len(voxel_ratio.loc[voxel_ratio[target]==0]))
print('===========================================================================')
p_list = []
for region in ['wt', 'ne', 'ce']:
    print(f'{region}_volume')
    X = voxel_ratio[[f'{region}_volume']]  
    low = voxel_ratio.loc[voxel_ratio[target]==1, f'{region}_volume']
    high = voxel_ratio.loc[voxel_ratio[target]==0, f'{region}_volume']
    mann_u, mann_p = mannwhitneyu(low, high)
    if mann_p < treshold:
        print(f'    mann_u: {mann_u}, mann_p: {mann_p}')
        print(f'{target} mean', low.mean(), f'non-{target} mean', high.mean())
        X = sm.add_constant(X)
        logit_model = sm.Logit(y, X)
        result = logit_model.fit()
        y_pred_prob = result.predict(X)
        auc = roc_auc_score(y, y_pred_prob)
        print('AUC', round(auc, 3))
        print()
    for cluster in range(n_cluster): 
        for data in [f'{region}_{cluster}_volume']: 
            print(data)
            X = voxel_ratio[[data]]  
            low = voxel_ratio.loc[voxel_ratio[target]==1, data]
            high = voxel_ratio.loc[voxel_ratio[target]==0, data]
            mann_u, mann_p = mannwhitneyu(low, high)
            p_list.append(mann_p)
            if mann_p < treshold:
                print(f'    mann_u: {mann_u}, mann_p: {mann_p}')
                print(f'{target} mean', low.mean(), f'non-{target} mean', high.mean())
                X = sm.add_constant(X)
                logit_model = sm.Logit(y, X)
                result = logit_model.fit()
                y_pred_prob = result.predict(X)
                intercept = result.params[0]  # const
                coef = result.params[1]       # feature
                cutoff_x = -intercept / coef 
                auc = roc_auc_score(y, y_pred_prob)
                print('AUC', round(auc, 3), "decision boundary: ", round(cutoff_x, 3))
                print()

In [ ]:

target = 'decrease'
y = voxel_ratio[target]
print('===========================================================================')
print(target)
print(f'number of {target}: ', len(voxel_ratio.loc[voxel_ratio[target]==1]))
print(f'number of non-{target}: ', len(voxel_ratio.loc[voxel_ratio[target]==0]))
print('===========================================================================')
p_list = []
for region in ['wt', 'ne', 'ce']:
    print(f'{region}_volume')
    X = voxel_ratio[[f'{region}_volume']]  # 독립변수 # 
    low = voxel_ratio.loc[voxel_ratio[target]==1, f'{region}_volume']
    high = voxel_ratio.loc[voxel_ratio[target]==0, f'{region}_volume']
    mann_u, mann_p = mannwhitneyu(low, high)
    if mann_p < treshold:
        print(f'    mann_u: {mann_u}, mann_p: {mann_p}')
        print(f'{target} mean', low.mean(), f'non-{target} mean', high.mean())
        X = sm.add_constant(X)
        logit_model = sm.Logit(y, X)
        result = logit_model.fit()
        y_pred_prob = result.predict(X)
        auc = roc_auc_score(y, y_pred_prob)
        print('AUC', round(auc, 3))
        print()
    for cluster in range(n_cluster): 
        for data in [f'{region}_{cluster}_volume']: 
            print(data)
            X = voxel_ratio[[data]]
            low = voxel_ratio.loc[voxel_ratio[target]==1, data]
            high = voxel_ratio.loc[voxel_ratio[target]==0, data]
            mann_u, mann_p = mannwhitneyu(low, high)
            p_list.append(mann_p)
            if mann_p < treshold:
                print(f'    mann_u: {mann_u}, mann_p: {mann_p}')
                print(f'{target} mean', low.mean(), f'non-{target} mean', high.mean())
                X = sm.add_constant(X)
                logit_model = sm.Logit(y, X)
                result = logit_model.fit()
                y_pred_prob = result.predict(X)
                intercept = result.params[0]  # const
                coef = result.params[1]       # feature
                cutoff_x = -intercept / coef 
                auc = roc_auc_score(y, y_pred_prob)
                fpr, tpr, thresholds = roc_curve(y, y_pred_prob)
                print('AUC', round(auc, 3), "decision boundary: ", round(cutoff_x, 3))
                print()

In [ ]:

pre_total = pd.DataFrame()
for subj in tqdm(sorted(original_meta.index)):
    directory = os.path.join(data_dir, subj, '2')
    if not os.path.isdir(directory): continue

    df = pd.DataFrame()
    for file_name, name in zip(['flair_vox.nii.gz', 'rCBV_vox.nii.gz', 'adc_vox.nii.gz', 't1ce_vox.nii.gz', 
                                'flair_rad.nii.gz', 'rCBV_rad.nii.gz', 'adc_rad.nii.gz', 't1ce_rad.nii.gz', 
                                ], 
                         ['flair', 'cbv', 'adc', 't1ce', 
                          'flair_normed', 'cbv_normed', 'adc_normed', 't1ce_normed', 
                          ]):
        img = ants.image_read(os.path.join(directory, file_name))
        img = img.numpy()
        img = img.flatten()
        df = pd.concat([df, pd.DataFrame({name:img})], axis=1)

    for file_name, name in zip(['NE_tumor_mask.nii.gz', 'CE_tumor_mask.nii.gz'], ['NE', 'CE']):
        img = ants.image_read(os.path.join(directory, file_name))
        img = img.numpy()
        img = img.flatten()
        img = pd.Series(img)
        img = img[img>0].index
        df[name] = None
        df.loc[img, name] = img

    df = df[(~pd.isnull(df['NE'])) | (~pd.isnull(df['CE']))]
    df['subj'] = subj
    df['flair_adc'] = (df['flair_normed']+10000) / (df['adc_normed']+10000)
    df['cbv_adc'] = (df['cbv_normed']+10000) / (df['adc_normed']+10000)

    pre_total = pd.concat([pre_total, df], axis=0)
    
pre_total['NE1CE0'] = pre_total.apply(lambda row: 1 if row['NE'] is not None else 0, axis=1)
# leave only valid voxels
pre_total = pre_total[(pre_total['adc']>0)&(pre_total['cbv']>0)&(pre_total['flair']>0)&(pre_total['t1ce']>0)]
# 99.9 percentile as outliers
percentiles = pre_total[['adc', 'cbv', 'flair', 't1ce', 'flair_adc', 'cbv_adc']].quantile(0.999)
# remove outlier
pre_total = pre_total[
    (pre_total['adc'] <= percentiles['adc']) &
    (pre_total['cbv'] <= percentiles['cbv']) &
    (pre_total['flair'] <= percentiles['flair']) &
    (pre_total['t1ce'] <= percentiles['t1ce']) # &
]

pre_total['wt_kmeans'] = kmeans.predict(pre_total[['cbv_normed', 'adc_normed']])
sns.scatterplot(data=pre_total, x='cbv_normed', y='adc_normed', hue='wt_kmeans', s=1, palette=palette)
plt.title(f'Recurred whole tumor clustered based on the initial tumor. {len(pre_total)} voxels')
plt.xlabel('nrCBV')
plt.ylabel('nADC')
plt.legend(markerscale=5)
plt.plot([common_range['cbv_min'], common_range['cbv_min'], common_range['cbv_max'], common_range['cbv_max'], common_range['cbv_min']], 
         [common_range['adc_min'], common_range['adc_max'], common_range['adc_max'], common_range['adc_min'], common_range['adc_min']], 
         'r-', 
         linewidth=2)
plt.xlim(total_range['cbv_min']-50, total_range['cbv_max']+50)
plt.ylim(total_range['adc_min']-50, total_range['adc_max']+50)

total['recur'] = pre_total.copy()

In [ ]:
row, col = 1, 2  
fig, ax = plt.subplots(nrows=row, ncols=col, figsize=(col*7, row*5))
plt.subplots_adjust(wspace=0.4, hspace=0.4)

xlim = (total_range['cbv_min'], total_range['cbv_max'])
ylim = (total_range['adc_min'], total_range['adc_max'])

subjs = ['initial', 'recur']
titles = ['Initial', 'Recur']

for i, subj in enumerate(subjs):
    pre_to_plot = total[subj].copy()
    pre_to_plot['wt_kmeans_named'] = pre_to_plot['wt_kmeans'].map({0:'Nonviable', 
                                                                   1: 'Hypovascular cellular',
                                                                   2: 'Hypervascular cellular'})
    sns.scatterplot(
        data=pre_to_plot,
        x='cbv_normed',
        y='adc_normed',
        s=1,
        hue='wt_kmeans_named',
        hue_order=['Nonviable', 'Hypovascular cellular', 'Hypervascular cellular'],
        palette={'Nonviable':palette[0], 'Hypovascular cellular':palette[1], 'Hypervascular cellular':palette[2]},
        ax=ax[i]
    ).set(title=f'{titles[i]} whole tumor clustering {len(pre_to_plot)} voxels', xlim=xlim, ylim=ylim)
    ax[i].legend(markerscale=5)
    ax[i].set_xlabel('nrCBV')
    ax[i].set_ylabel('nADC')

plt.subplots_adjust(hspace=0.2, wspace=0.3) 
plt.show()


In [ ]:
pre_total = total['recur'].copy()

voxel_ratio = pd.DataFrame()

for subj in tqdm(sorted(original_meta.index)):
    directory = os.path.join(data_dir, subj, '2')
    if not os.path.isdir(directory): continue

    rta = survival.loc[subj].RTA
    rta_cont = survival.loc[subj].RTA_cont
    ta1 = survival.loc[subj].TA1
    sex = survival.loc[subj].sex
    age = survival.loc[subj].age
    
    if sex == 'M':
        sex = 1
    else: # age == 'F':
        sex = 0

    if rta == 'D':
        increase=0
        decrease=1
    elif rta == 'I':
        increase=1
        decrease=0
    else: # if rta == 'S':
        increase=0
        decrease=0

    subj_df = pre_total[pre_total['subj']==subj].copy() 

    dictionary = {'RTA':rta, 'RTA_cont':rta_cont, 'TA1':ta1, 'increase':increase, 'decrease':decrease, 'Sex':sex, 'Age':age}
    for region in ['wt', 'ne', 'ce']:
        for cluster in range(n_cluster):
            if region == 'wt':
                dictionary[f'{region}_voxel_num'] = len(subj_df)
                dictionary[f'{region}_{cluster}_voxel_num'] = len(subj_df[subj_df['wt_kmeans'] == cluster])
                dictionary[f'{region}_{cluster}'] = dictionary[f'{region}_{cluster}_voxel_num']/dictionary[f'{region}_voxel_num']
            elif region == 'ne':
                dictionary[f'{region}_voxel_num'] = len(subj_df[subj_df['NE1CE0']==1])
                dictionary[f'{region}_{cluster}_voxel_num'] = len(subj_df[(subj_df['NE1CE0']==1)&(subj_df['wt_kmeans'] == cluster)])
                dictionary[f'{region}_{cluster}'] = dictionary[f'{region}_{cluster}_voxel_num']/dictionary[f'{region}_voxel_num']
            elif region == 'ce':
                dictionary[f'{region}_voxel_num'] = len(subj_df[subj_df['NE1CE0']==0])
                dictionary[f'{region}_{cluster}_voxel_num'] = len(subj_df[(subj_df['NE1CE0']==0)&(subj_df['wt_kmeans'] == cluster)])
                dictionary[f'{region}_{cluster}'] = dictionary[f'{region}_{cluster}_voxel_num']/dictionary[f'{region}_voxel_num']

    voxel_ratio = pd.concat([voxel_ratio, pd.DataFrame(dictionary, index=[int(subj)])], axis=0)

output = pd.DataFrame()
for sd in [f'{data_dir}/{pid}/2' for pid in survival.index]:
    df = calc_volume_ml(sd)
    output = pd.concat([output, df], axis=0)

output = output[['brain_volume', 'wt_volume', 'ce_volume', 'ne_volume']]

output.index = output.index.astype(int)
voxel_ratio = voxel_ratio.join(output, how='left')
voxel_ratio.index.name = 'ID'
habitat_ratio['recur'] = voxel_ratio.copy()

# Participant-level table display omitted for public distribution.

In [ ]:
for op in ['initial', 'recur']:
    voxel_ratio = habitat_ratio[op].copy()
    for region in ['wt', 'ne', 'ce']:
        for cluster in range(n_cluster):
            voxel_ratio[f'{region}_{cluster}_volume'] = voxel_ratio[f'{region}_{cluster}'] * voxel_ratio[f'{region}_volume']
            voxel_ratio[f'{region}_{cluster}_adj'] = voxel_ratio[f'{region}_{cluster}_volume'] / voxel_ratio['brain_volume']
    habitat_ratio[op] = voxel_ratio.copy()



voxel_ratio = habitat_ratio['initial'].copy()
for region in ['wt', 'ne', 'ce']:
    for cluster in range(n_cluster):
        for data in [f'{region}_{cluster}', f'{region}_{cluster}_voxel_num', f'{region}_{cluster}_volume', f'{region}_volume', f'{region}_{cluster}_adj']:
            voxel_ratio[data] = habitat_ratio['recur'][data] - habitat_ratio['initial'][data]
voxel_ratio['brain_volume'] = habitat_ratio['recur']['brain_volume'] - habitat_ratio['initial']['brain_volume']
habitat_ratio['delta'] = voxel_ratio.copy()

In [ ]:
# Participant-level table display omitted for public distribution.

In [ ]:
voxel_ratio = habitat_ratio['delta'].dropna()
# Participant-level table display omitted for public distribution.

In [ ]:
print('spearmanr', spearmanr(voxel_ratio['ne_1_volume'], voxel_ratio['ne_volume'])) 

In [ ]:
print('spearmanr', spearmanr(voxel_ratio['ne_1_volume'], voxel_ratio['ce_volume'])) 

In [ ]:
print('spearmanr', spearmanr(voxel_ratio['ce_1_volume'], voxel_ratio['ce_volume']))

In [ ]:
print('spearmanr', spearmanr(voxel_ratio['ce_2_volume'], voxel_ratio['ce_volume']))

In [ ]:
print('spearmanr', spearmanr(voxel_ratio['ce_0_volume'], voxel_ratio['ce_volume']))

In [ ]:
print('spearmanr', spearmanr(voxel_ratio['ce_2_volume'], voxel_ratio['ne_volume']))

In [ ]:
print('spearmanr', spearmanr(voxel_ratio['ce_0_volume'], voxel_ratio['ne_volume']))

In [ ]:
print('spearmanr', spearmanr(voxel_ratio['ne_0_volume'], voxel_ratio['ce_volume'])) 

# Radiomics Analysis

In [ ]:
# change settings here
rad_root = f"{root}/excel_csv/radiomics_samespacing_2-2-6_new_bin5_initial_GTR" # the directory where radiomics data directories are located 
mask_type_list = ['NE', 'CE', 'WT']
modal_dir = {'ADC':'ADC_radiomics', 
            'CBV':'CBV_radiomics', 
            'T1CE':'T1CE_radiomics', 
            'FLAIR':'FLAIR_radiomics'}
sub_dirs = {'pre':'1'} 
radiomics = pd.DataFrame()

for mask_type in mask_type_list:
    shape_feature={'pre':None, 'post':None}
    for modal, directory in modal_dir.items():
        df={}
        for op, sub in sub_dirs.items():
            df[op] = f'{mask_type}2{modal}_{sub}_radiomics.csv'
            df[op] = pd.read_csv(os.path.join(rad_root, directory, sub, df[op]))
            df[op].set_index('subj', inplace=True) 
            if shape_feature[op] is None:
                shape_columns = list(filter(lambda col: col.startswith('original_shape_'), df[op].columns))
                shape_feature[op] = df[op][shape_columns]
            df[op] = df[op].drop(shape_columns, axis=1)
        df['pre'] = df['pre'].add_prefix(f'{mask_type}_{modal.lower()}_initial_')
        radiomics = pd.concat([radiomics, df['pre']], axis=1)
    shape_feature['pre'] = shape_feature['pre'].add_prefix(f'{mask_type}_shape_initial_')
    radiomics = pd.concat([radiomics, shape_feature['pre']], axis=1)

columns = radiomics.columns
columns

In [ ]:
radiomics.index = radiomics.index.astype(str)
radiomics = original_meta.join(radiomics, how='left')

In [ ]:
radiomics_null_df = radiomics.isnull().sum().reset_index()
radiomics_null_df.columns = ['Feature', 'Null_Count']
radiomics_null_df['Null_Count'].sum()

In [ ]:
radiomics_null_df

In [ ]:
rad_pvals = {}

In [ ]:
p_val_df = pd.DataFrame()
for col in columns: 
    if radiomics[col].nunique() == 1: 
        print('only one straight value', col)
        continue
    if radiomics[col].isna().sum() != 0: 
        print('there are NaN values', col)
        continue
    if np.isinf(radiomics[col]).sum() != 0: 
        print('there are infinite values', col)
        continue

    y = radiomics['TA1']
    spearman_rho, spearman_p = spearmanr(y, radiomics[col])
    kendall_tau, kendall_p = kendalltau(y, radiomics[col])
    mann_u, mann_p = mannwhitneyu(radiomics.loc[radiomics['TA1']>35, col], radiomics.loc[radiomics['TA1']<=35, col])
    name = col.split('_')
    p_val_df = pd.concat([p_val_df, pd.DataFrame({'region':name[0],
                                                'modal':name[1],
                                                'op':name[2],
                                                'filter':name[3],
                                                'Parent':name[4],
                                                'feature':'_'.join(name[5:]),
                                                'spearman_rho':spearman_rho, 
                                                'spearman_p':spearman_p, 
                                                'kendall_tau':kendall_tau, 
                                                'kendall_p':kendall_p, 
                                                'mann_u':mann_u, 
                                                'mann_p':mann_p, 
                                                }, index=[col])], axis=0)
p_val_df.head()

In [ ]:
print(len(p_val_df[p_val_df["region"]!="WT"]))
print(len(p_val_df[(p_val_df['modal']=='adc')&(p_val_df["region"]!="WT")]))
print(len(p_val_df[(p_val_df['modal']=='cbv')&(p_val_df["region"]!="WT")]))
print(len(p_val_df[(p_val_df['modal']=='flair')&(p_val_df["region"]!="WT")]))
print(len(p_val_df[(p_val_df['modal']=='t1ce')&(p_val_df["region"]!="WT")]))

In [ ]:
p_val_df = p_val_df[p_val_df["region"]!="WT"]

In [ ]:
criteria = "spearman_p"
# FDR (Benjamini-Hochberg) 
print('After FDR')
for seq in ['t1ce', 'flair', 'adc', 'cbv']:
    test = p_val_df.loc[(p_val_df['modal']==seq)] 
    reject2, pvals_corrected2, _, _ = multipletests(test[criteria], alpha=0.05, method='fdr_bh')
    print(f"(p_val_df['modal']=={seq}): ", np.sum(reject2))

In [ ]:
seq = 'adc'
test = p_val_df.loc[(p_val_df['modal']==seq)].copy() 
reject2, pvals_corrected2, _, _ = multipletests(test[criteria], alpha=0.05, method='fdr_bh')
print(f"(p_val_df['modal']=={seq}): ", np.sum(reject2))
test['fdr'] = pvals_corrected2
test[test['fdr']<0.05]

In [ ]:
test[(test['fdr']<0.05)&(test['region']=='NE')&(test['spearman_rho']>0)]

In [ ]:
print(test[(test['fdr']<0.05)&(test['region']=='NE')&(test['spearman_rho']>0)].spearman_rho.mean())
print(test[(test['fdr']<0.05)&(test['region']=='NE')&(test['spearman_rho']<0)].spearman_rho.mean())

In [ ]:
test[(test['fdr']<0.05)&(test['spearman_rho']<0)]

In [ ]:
rad_pvals['TA1'] = p_val_df.copy()

In [ ]:
p_val_df = pd.DataFrame()
for col in list(set([c[3:] for c in columns])): 
    x = radiomics[f'NE_{col}']-radiomics[f'CE_{col}']
    if x.nunique() == 1: 
        print('only one straight value', col)
        continue
    if x.isna().sum() != 0: 
        print('there are NaN values', col)
        continue
    if np.isinf(x).sum() != 0: 
        print('there are infinite values', col)
        continue

    y = radiomics['RTA_cont'] # TA2 - TA1
    spearman_rho, spearman_p = spearmanr(y, x)
    kendall_tau, kendall_p = kendalltau(y, x)
    x = radiomics[['decrease', f'NE_{col}', f'CE_{col}']].copy()
    x['col'] = x[f'NE_{col}'] - x[f'CE_{col}']
    mann_u, mann_p = mannwhitneyu(x.loc[x['decrease']==1, 'col'], x.loc[x['decrease']!=0, 'col'])
    roc_auc=None

    df = x[['col']]
    df = sm.add_constant(df)
    y = x['decrease']
    model = sm.Logit(y, df)
    result = model.fit(disp=0) 
    y_pred_prob = result.predict(df)
    roc_auc = roc_auc_score(y, y_pred_prob)

    name = col.split('_')
    p_val_df = pd.concat([p_val_df, pd.DataFrame({
                                                'modal':name[0],
                                                'op':name[1],
                                                'filter':name[2],
                                                'Parent':name[3],
                                                'feature':'_'.join(name[4:]),
                                                'spearman_rho':spearman_rho, 
                                                'spearman_p':spearman_p, 
                                                'kendall_tau':kendall_tau, 
                                                'kendall_p':kendall_p, 
                                                'mann_u':mann_u, 
                                                'mann_p':mann_p, 
                                                "ROC AUC": roc_auc,
                                                }, index=[col])], axis=0)
p_val_df.head()

In [ ]:
print(len(p_val_df))
print(len(p_val_df[p_val_df['modal']=='adc']))
print(len(p_val_df[p_val_df['modal']=='cbv']))
print(len(p_val_df[p_val_df['modal']=='flair']))
print(len(p_val_df[p_val_df['modal']=='t1ce']))

In [ ]:
criteria = "spearman_p"

In [ ]:
# FDR (Benjamini-Hochberg) 
print('After FDR')
for seq in ['t1ce', 'flair', 'adc', 'cbv']:
    test = p_val_df.loc[(p_val_df['modal']==seq)] #  & (p_val_df['filter']=='original')  & (p_val_df['region']=='NE') & (p_val_df['Parent']=='firstorder')
    reject2, pvals_corrected2, _, _ = multipletests(test[criteria], alpha=0.05, method='fdr_bh')
    print(f"(p_val_df['modal']=={seq}): ", np.sum(reject2))

In [ ]:
seq = 'adc'
test = p_val_df.loc[(p_val_df['modal']==seq)].copy() 
reject2, pvals_corrected2, _, _ = multipletests(test[criteria], alpha=0.05, method='fdr_bh')
print(f"(p_val_df['modal']=={seq}): ", np.sum(reject2))
test['fdr'] = pvals_corrected2
test[test['fdr']<0.05]

In [ ]:
len(test[(test['fdr']<0.05)&(test['feature']=='90Percentile')])

In [ ]:
print(len(test[(test['fdr']<0.05)&(test['filter']=='exponential')&(test['spearman_rho']>0)]))
print(test[(test['fdr']<0.05)&(test['filter']=='exponential')&(test['spearman_rho']>0)].spearman_rho.mean())
print(len(test[(test['fdr']<0.05)&(test['filter']=='exponential')&(test['spearman_rho']<0)]))
print(test[(test['fdr']<0.05)&(test['filter']=='exponential')&(test['spearman_rho']<0)].spearman_rho.mean())

In [ ]:
seq = 't1ce'
test = p_val_df.loc[(p_val_df['modal']==seq)].copy()  
reject2, pvals_corrected2, _, _ = multipletests(test[criteria], alpha=0.05, method='fdr_bh')
print(f"(p_val_df['modal']=={seq}): ", np.sum(reject2))
test['fdr'] = pvals_corrected2
test[test['fdr']<0.05]